<a href="https://colab.research.google.com/github/Eton-np/DataMining/blob/main/stationeryipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message=".*utcnow.*",
    category=DeprecationWarning
)

In [ ]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori,association_rules

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/DataSet/datasetC_stationery_8.csv")
df.shape
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   bill_id  20 non-null     int64 
 1   item     20 non-null     object
dtypes: int64(1), object(1)
memory usage: 452.0+ bytes


,bill_id,item
0,1,Pen
1,1,Notebook
2,1,Ruler
3,2,Pen
4,2,Notebook


In [ ]:
#แปลงข้อมูลให้อยู่ในรูปแบบของ Basket
basket = (df.groupby(['bill_id','item'])['item']
          .count()
          .unstack()
          .reset_index()
          .fillna(0)
          .set_index('bill_id')
          )
basket

item,Eraser,Notebook,Pen,Ruler
bill_id,,,,
1,0.0,1.0,1.0,1.0
2,0.0,1.0,1.0,0.0
3,1.0,0.0,1.0,0.0
4,0.0,1.0,0.0,1.0
5,1.0,1.0,1.0,0.0
6,0.0,0.0,1.0,1.0
7,1.0,1.0,0.0,1.0
8,0.0,1.0,1.0,1.0


In [ ]:
#แปลงจากตัวเลขให้เป็น Boolean
basket_encoded = basket.map(lambda x: x>0)
basket_encoded

item,Eraser,Notebook,Pen,Ruler
bill_id,,,,
1,False,True,True,True
2,False,True,True,False
3,True,False,True,False
4,False,True,False,True
5,True,True,True,False
6,False,False,True,True
7,True,True,False,True
8,False,True,True,True


In [ ]:
#หาชุดสินค้าที่ซื้อร่วมกันบ่อยๆ
frequent_itemset = apriori(basket_encoded,min_support=0.3,use_colnames=True)
#สร้างกฏความสัมพันธ์
rules = association_rules(frequent_itemset,metric='lift',min_threshold=1)
rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Ruler),(Notebook),0.625,0.750,0.5,0.800000,1.066667,1.0,0.03125,1.250,0.166667,0.571429,0.200000,0.733333
1,(Notebook),(Ruler),0.750,0.625,0.5,0.666667,1.066667,1.0,0.03125,1.125,0.250000,0.571429,0.111111,0.733333


In [ ]:
#คัดเฉพาะกฏที่สำคัญ
rules_sort = rules[['antecedents','consequents','support','confidence','lift']].sort_values(
    by=['lift','confidence'],ascending = False
)
print("กฏความสัมพันธ์ที่ค้นพบ")
print(rules_sort.to_string())

กฏความสัมพันธ์ที่ค้นพบ
  antecedents consequents  support  confidence      lift
0     (Ruler)  (Notebook)      0.5    0.800000  1.066667
1  (Notebook)     (Ruler)      0.5    0.666667  1.066667
